In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable
from pyspark.sql.window import Window
import uuid
import hashlib

In [0]:
%run /Workspace/DE_Project_Pipeline/1_setup/utilities

In [0]:
print(bronze_schema, silver_schema, gold_schema)

In [0]:
dbutils.widgets.text("catalog", "fmcg_de_project", "Catalog")
dbutils.widgets.text("data_source", "orders", "Data Source")

catalog = dbutils.widgets.get("catalog")
data_source = dbutils.widgets.get("data_source")

# base_path = f's3://sportsbar-fmcg-de-project-viswadithya/{data_source}'
# landing_path=f"{base_path}/landing/"
# processed_path=f"{base_path}/processed/"
# print("base_path: ", base_path)
# print("landing_path: ", landing_path)
# print("processed_path: ", processed_path)


bronze_table=f"{catalog}.{bronze_schema}.{data_source}"
silver_table=f"{catalog}.{silver_schema}.{data_source}"
gold_table=f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
# df = spark.read.options(header=True, inferSchema=True).csv(f"{landing_path}/*.csv").withColumn("read_timestamp", F.current_timestamp()).select("*", "_metadata.file_name", "_metadata.file_size")

# print("Total Rows: ", df.count())
# df.show(5)

In [0]:
# display(df.limit(10))

In [0]:
print(bronze_table)

###COALESCE

In [0]:
# from pyspark.sql import functions as F

# # Step 1: Read all CSV files belonging to the batch
# # df_bronze = spark.read \
# #     .option("header", True) \
# #     .option("inferSchema", True) \
# #     .csv(batch_path)

# # Step 2: Add your existing metadata columns here
# # file_name, file_size, read_timestamp, batch_id

# # Step 3: Calculate total input file size
# file_stats = (
#     df_bronze
#     .select("file_name", "file_size")
#     .dropDuplicates(["file_name"])
#     .agg(F.sum("file_size").alias("total_bytes"))
#     .first()
# )

# total_bytes = file_stats["total_bytes"] or 0. #Falsy value

# # Step 4: Control partitions before writing
# if total_bytes < 128 * 1024 * 1024:  #128 MB
#     df_bronze = df_bronze.coalesce(1)
# else:
#     df_bronze = df_bronze.repartition(20)

# # Step 5: Write into Bronze Delta
# df_orders.write \
#     .format("delta") \
#     .partitionBy("order_month") \
#     .mode("overwrite") \
#     .saveAsTable("fmcg_de_project.bronze.orders_partitioned")

In [0]:
# spark.sql(OPTIMIZE fmcg_de_project.bronze.orders_partitioned
# WHERE order_month = '2026-01';)

In [0]:
# df.write\
#     .format("delta")\
#         .option("delta.enableChangeDataFeed", True)\
#             .mode("append")\
#                 .saveAsTable(bronze_table)


### Batch ID and JOB ID

In [0]:
# files=dbutils.fs.ls(landing_path)

# files=[x for x in files if not x.isDir()]

# if not files:
#     raise ValueError("No files found in landing folder")

# file_signature="|".join(
#     sorted(
#         f"{x.path}:{x.size}:{x.modificationTime}"
#         for x in files
#     )
# )

# batch_hash = hashlib.sha256(
#     file_signature.encode()
# ).hexdigest()[:12]

# batch_id = f"{data_source}_{batch_hash}"

# job_id = str(uuid.uuid4())

# print("Batch ID:", batch_id)
# print("Job ID:", job_id)

In [0]:
# files = dbutils.fs.ls(landing_path)
# for file_info in files:
#     dbutils.fs.mv(
#         file_info.path,
#         f"{processed_path}/{file_info.name}",
#         True
#     )

In [0]:
# batch_id = f"{data_source}_{batch_hash}"

In [0]:
df_bronze = spark.sql(f"SELECT * FROM {bronze_table}")
df_bronze.show(2)

display(df_bronze.count())

In [0]:
df_bronze.explain("formatted")

In [0]:
detail = spark.sql("DESCRIBE DETAIL fmcg_de_project.bronze.orders")
display(detail)

In [0]:
# df_bronze.rdd.getNumPartitions()

In [0]:
df_invalid_records=df_bronze.filter(
    F.col("order_id").isNull() |
    F.col("order_placement_date").isNull() |
    F.col("customer_id").isNull() |
    F.col("product_id").isNull()
)

In [0]:
df_invalid_records=df_invalid_records.withColumn(
    "Rejection_Reason",
    F.when(F.col("order_id").isNull(), "Missing order id")\
    .when(F.col("order_placement_date").isNull(), "Missing order_placement_date")\
    .when(F.col("customer_id").isNull(), "Missing customer_id")\
    .when(F.col("product_id").isNull(), "Missing product_id")
)

In [0]:
display(df_invalid_records)

###Quarantine

In [0]:
df_invalid_records.write\
    .format("delta")\
        .mode("append")\
            .option("mergeSchme", True)\
                .saveAsTable(f"{catalog}.{silver_schema}.quarantine_{data_source}")

In [0]:
# spark.sql(f"""
#     DROP TABLE IF EXISTS {catalog}.{silver_schema}.quarantine_{data_source}
# """)

In [0]:
df_valid_records=df_bronze.filter(
    (F.col("order_id").isNotNull()) &
    (F.col("order_placement_date").isNotNull()) &
    (F.col("customer_id").isNotNull()) &
    (F.col("product_id").isNotNull())
)

In [0]:
# print(df_orders.columns)

In [0]:
display(df_valid_records.limit(4))

In [0]:
# Cdc_timestamp="2025-11-30 23:59:59"

# df_valid_records=df_valid_records.filter(
#     F.col("read_timestamp")<=F.to_timestamp(F.lit(Cdc_timestamp))
# )

# display(df_valid_records.limit(4))

In [0]:
df_orders=df_valid_records
# 1. Keep only rows where order_qty is present
# df_orders = df_orders.filter(F.col("order_qty").isNotNull())


# 2. Clean customer_id → keep numeric, else set to 999999
df_orders = df_orders.withColumn(
    "customer_id",
    F.when(F.col("customer_id").rlike("^[0-9]+$"), F.col("customer_id"))
     .otherwise("999999")
     .cast("string")
)

# 3. Remove weekday name from the date text
#    "Tuesday, July 01, 2025" → "July 01, 2025"
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.regexp_replace(F.col("order_placement_date"), r"^[A-Za-z]+,\s*", "")
)

# 4. Parse order_placement_date using multiple possible formats
df_orders = df_orders.withColumn(
    "order_placement_date",
    F.coalesce(
        F.try_to_date("order_placement_date", "yyyy/MM/dd"),
        F.try_to_date("order_placement_date", "dd-MM-yyyy"),
        F.try_to_date("order_placement_date", "dd/MM/yyyy"),
        F.try_to_date("order_placement_date", "MMMM dd, yyyy"),
    )
)

# 5. Drop duplicates
# df_orders = df_orders.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])

# 5. convert product id to string
df_orders = df_orders.withColumn('product_id', F.col('product_id').cast('string'))


### removing duplicates

In [0]:
w=Window.partitionBy("order_id","order_placement_date","customer_id","product_id","order_qty","read_timestamp").orderBy("order_placement_date")

df_orders=df_orders.withColumn(
    "rn",
    F.row_number().over(w)
)

In [0]:
display(df_orders.limit(4))

In [0]:
df_orders=df_orders.filter(F.col("rn")==1)

In [0]:
display(df_orders.limit(4))

### LATE ARRIVING RECORDS

In [0]:
Cdc_timestamp="2025-12-01"

df_orders=df_orders.filter(
    F.col("order_placement_date")<F.to_timestamp(F.lit(Cdc_timestamp))
)

display(df_orders.limit(4))

In [0]:
print(df_orders.printSchema())

In [0]:
print("Total Bronze records", df_bronze.count())
print("Total invalid records", df_invalid_records.count())
print("Total valid records", df_orders.count())

### Salting

In [0]:
df_orders_group=df_orders.groupBy("product_id").count().orderBy(F.col("count").desc())

display(df_orders_group)

In [0]:
df_orders.groupBy("product_id") \
    .count() \
    .agg(
        F.min("count").alias("min_count"),
        F.max("count").alias("max_count"),
        F.avg("count").alias("avg_count")
    ) \
    .show()

In [0]:
total_count = df_orders.count()

hot_keys = (
    df_orders.groupBy("product_id")
    .count()
    .withColumn(
        "percentage",
        F.col("count") * 100 / total_count
    )
    .filter(
        (F.col("count") >= 2600) |
        (F.col("percentage") > 30)
    )
)

hot_keys.show()

In [0]:
hot_fact = df_orders.join(
    hot_keys,
    on="product_id",
    how="inner"
)

normal_fact = df_orders.join(
    hot_keys,
    on="product_id",
    how="left_anti"
)

In [0]:
num_salts = 10

hot_fact = hot_fact.withColumn(
    "salt",
    F.pmod(
        F.xxhash64("order_id"),
        F.lit(num_salts)
    )
)

In [0]:
df_products = spark.table(f"{catalog}.{silver_schema}.products_non_aws_new")
display(df_products.limit(5))

In [0]:
hot_product = df_product.join(
    hot_keys,
    on="product_id",
    how="inner"
)

hot_product = hot_product.withColumn(
    "salt",
    F.explode(
        F.sequence(
            F.lit(0),
            F.lit(num_salts - 1)
        )
    )
)

In [0]:
hot_joined = hot_fact.join(
    hot_product,
    on=["product_id", "salt"],
    how="left"
).drop("salt")

In [0]:
normal_joined = normal_fact.join(
    df_product,
    on="product_id",
    how="left"
)

In [0]:
df_joined = normal_joined.unionByName(
    hot_joined
)

In [0]:
# check what's the maximum and minimum date
df_orders.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
display(df_orders.limit(10))

In [0]:
df_products = spark.table(f"{catalog}{silver_schema}.{data_source}")
display(df_products.limit(5))

In [0]:
# df_joined = df_orders.join(df_products, on="product_id", how="inner").select(df_orders["*"], df_products["product_code"])

# display(df_joined.limit(5))

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_joined.write.format("delta").option(
        "delta.enableChangeDataFeed", "true"
    ).option("mergeSchema", "true").mode("overwrite").saveAsTable(silver_table)
else:
    silver_delta = DeltaTable.forName(spark, silver_table)
    silver_delta.alias("silver").merge(df_joined.alias("bronze"), "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# df_old = (
#     spark.read.format("delta")
#     .option("versionAsOf", 2)
#     .table(silver_table)
# )

In [0]:
# df_old = (
#     spark.read.format("delta")
#     .option("timestampAsOf", "2026-11-06 10:00:00")
#     .table(silver_table)
# )

In [0]:
# RESTORE TABLE fmcg_de_project.silver.fact_sales
# TO VERSION AS OF 2;

In [0]:
# VACUUM fmcg_de_project.silver.fact_sales;

In [0]:
# cdf_df = (
#     spark.read.format("delta")
#     .option("readChangeFeed", "true")
#     .option("startingVersion", 1)
#     .option("endingVersion", 3)
#     .table(silver_table)
# )

In [0]:
# df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code, product_code, product_id, order_qty as sold_quantity FROM {silver_table};")

# df_gold.show(2)

In [0]:
# if not (spark.catalog.tableExists(gold_table)):
#     print("creating New Table")
#     df_gold.write.format("delta").option(
#         "delta.enableChangeDataFeed", "true"
#     ).option("mergeSchema", "true").mode("overwrite").saveAsTable(gold_table)
# else:
#     gold_delta = DeltaTable.forName(spark, gold_table)
#     gold_delta.alias("source").merge(df_gold.alias("gold"), "source.date = gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

In [0]:
# df_child = spark.sql(f"SELECT date, product_code, customer_code, sold_quantity FROM {gold_table}")
# df_child.show(10)

In [0]:
# df_child.count()

In [0]:
# df_monthly = (
#     df_child
#     # 1. Get month start date (e.g., 2025-11-30 → 2025-11-01)
#     .withColumn("month_start", F.trunc("date", "MM"))   # or F.date_trunc("month", "date").cast("date")

#     # 2.Group at monthly grain by month_start + product_code + customer_code
#     .groupBy("month_start", "product_code", "customer_code")
#     .agg(
#         F.sum("sold_quantity").alias("sold_quantity")
#     )

#     # 3. Rename month_start back to `date` to match your target schema
#     .withColumnRenamed("month_start", "date")
# )

# df_monthly.show(5, truncate=False)

In [0]:
# df_monthly.count()

In [0]:
# gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
# gold_parent_delta.alias("parent_gold").merge(df_monthly.alias("child_gold"), "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()